# Regression test : capture cross-submission dans Z3.Linq (Polyglot Notebooks)

Ce notebook est un **test de regression** pour le bug de l'issue #2 : un theoreme dont une contrainte capture une variable declaree dans **une autre cellule** .NET Interactive (un autre "submission") levait `System.ArgumentException` lors du `.Solve()`.

## Cause racine

`ExpressionVisitor.VisitMember` (~L461) resout les constantes host-capturees par reflexion `field.GetValue(target)`. Dans .NET Interactive, chaque cellule compile en un assembly dynamique separe (`Submission#N`). Quand le lambda d'une contrainte capture un local declare dans une autre submission, le `FieldInfo` est resolu sur la submission du lambda tandis que le `target` (le closure) porte le type de la submission d'origine :

```
System.ArgumentException: Field 'b0' defined on type 'Submission#N' is not a field
on the target object of type 'Submission#M'
```

Ce bug etait invisible en xUnit (single-assembly) car la closure et le field sont dans le meme assembly.

## Fix

`Theorem.AssertConstraints` appelle `PartialEvaluator.PartialEval(constraint.Body, ...)` **avant** le `ExpressionVisitor.Visit`. Cela replie les constantes capturees en `ConstantExpression` litteraux, neutralisant la resolution reflexive cross-assembly. Les sous-arbres qui referencent le parametre du theoreme (ex. `t.Values[i]`) sont preserves (non-evaluables sans liaison de parametre).

## Prerequis

```
dotnet build solutions/Z3.Linq.sln
dotnet publish solutions/Z3.Linq/Z3.Linq.csproj -o .deploy
```

Ajustez le chemin absolu dans la cellule 1 (`SetCurrentDirectory`) selon votre checkout.

In [1]:
// Ajustez ce chemin absolu selon votre checkout du fork Z3.Linq.
System.IO.Directory.SetCurrentDirectory(@"D:\dev\CoursIA-2\MyIA.AI.Notebooks\SymbolicAI\SMT\Z3.Linq\solutions\polyglot-repro");
Console.WriteLine("Working directory configured.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Working directory configured.


In [2]:
#r "../../.deploy/Microsoft.Z3.dll"
#r "../../.deploy/ExpressionUtils.dll"
#r "../../.deploy/Z3.Linq.dll"

using Microsoft.Z3;
using Z3.Linq;

// Modele de theoreme : trois entiers.
public class Triple
{
    public int[] Values { get; set; } = new int[3];
}

// Bornes declarees dans CETTE submission (cellule).
var b0 = 7;
var b1 = 8;
var b2 = 9;
var ctx = new Z3Context();
Console.WriteLine("[setup] b0, b1, b2, ctx declares dans cette submission.");

[setup] b0, b1, b2, ctx declares dans cette submission.


## Cellule (submission suivante) - capture cross-submission

La contrainte ci-dessous capture `b0`, `b1`, `b2` (declares dans la submission precedente). **Avant le fix**, cela levait `ArgumentException`. **Apres le fix** (PartialEval dans `Theorem.AssertConstraints`), cela resout correctement.

In [3]:
var theorem = ctx.NewTheorem<Triple>()
    .Where(t => t.Values[0] == b0)
    .Where(t => t.Values[1] == b1)
    .Where(t => t.Values[2] == b2);

var solution = theorem.Solve();
Console.WriteLine($"[cross-cell] Solution = {solution.Values[0]}, {solution.Values[1]}, {solution.Values[2]}");
ctx.Dispose();

[cross-cell] Solution = 7, 8, 9
